### Bag of Stories
**Description:** Generate stories using Large Language Models (LLMs) and opening fragments from a selection of Grimm folktales. The code was created with Visual Studio Code (VSCode), GitHub Copilot support and Jupyter extension, under Windows 11.

**Contributor:** Florentina Armaselu  

Required libraries

In [ ]:
# Import required libraries
from llama_cpp import Llama
from pathlib import Path
import pandas as pd
import math
import datetime as dt
import os

Define the model and input, output paths

In [ ]:
# Define the base directory relative to the current script location
# The models should be previously downloaded and placed in the models folder
base_dir = os.getcwd()
# Set the path to the models folder
models = os.path.join(base_dir, "./../../../../huggingface/cli/models/")
# Set the path to the input data folder for human stories
input = os.path.join(base_dir, "./../data/human/grimm/")
# Set the path to the input data folder for segmented stories with text-tiling
input_seg_tt = os.path.join(base_dir, "./../data/annotated_stories/human/grimm/text-tiling/")
# Set the path to the input data folder for segmented stories with LLM episodes
# input_seg_llm = os.path.join(base_dir, "./../data/annotated_stories/human/grimm/llm-episodes/")
# Set the path to the output folder for SGS results on the input stories starting with equal length segments
output_el_start = os.path.join(base_dir, "./../data/ai-gen/grimm/el_start/")
# Set the path to the output folder for SGS results on the segmented stories starting with text-tiling segments
output_tt_start = os.path.join(base_dir, "./../data/ai-gen/grimm/tt_start/")
# Set the path to the output folder for SGS results on the segmented stories starting with LLM episodes
# output_llme_start = os.path.join(base_dir, "./../data/ai-gen/grimm/llme_start/")
# Set the path of the output folder for the intermediate results
output_inter_res = os.path.join(base_dir, "./../results/intermediate_results/grimm/")

Define constants.

In [ ]:
# String constants for segmentation methods
SEG_EQUAL_LENGTH = 'equal-length'
SEG_TEXT_TILING = 'text-tiling'
# SEG_LLM_EPISODE = 'llm-episode'

# String constants for annotation elements
ANN_SEG = 'Segment' # Label for segments
ANN_SEG_PAIR = 'Segment Pair' # Label for segment pairs
ANN_SEP = '-'*40 # Separator for segments

# String constants for file naming
FN_EQUAL_LENGTH = 'el'
FN_TEXT_TILING = 'tt'
# FN_LLM_EPISODE = 'llme'

# String constant for generation (GEN) labels
GEN_LABEL = 'GEN'

# String constants for types of intermediate results
INTER_RES_SEG = 'segment_check' # Intermediate results for segment check

Load mistral-7b-instruct-v0.2.Q4_K_M.gguf (to be run separately)

In [ ]:
# Adjust the path to where your GGUF model is located
model_path = os.path.join(models, "mistral-7b-instruct-v0.2.Q4_K_M.gguf")

llm = Llama(
    model_path=model_path,
    n_ctx=8192,
    n_threads=8,
    n_gpu_layers=20  # set to 0 if using CPU only
)

Load llama-pro-8b-instruct.Q4_K_M.gguf (to be run separately)

In [ ]:
# Adjust the path to where your GGUF model is located
model_path = os.path.join(models, "llama-pro-8b-instruct.Q4_K_M.gguf")

llm = Llama(
    model_path=model_path,
    n_ctx=8192,
    n_threads=8,
    n_gpu_layers=20  # set to 0 if using CPU only
)

Load SOLAR-10.7B-Instruct-v1.0.Q4_K_M.gguf (to be run separately)

In [ ]:
# Adjust the path to where your GGUF model is located
model_path = os.path.join(models, "SOLAR-10.7B-Instruct-v1.0.Q4_K_M.gguf")

llm = Llama(
    model_path=model_path,
    n_ctx=8192,
    n_threads=8,
    n_gpu_layers=20  # set to 0 if using CPU only
)

Define the prompt and the function for generating the llm response using chunking

In [ ]:
# The prompt instructs the LLM to generate a story with a given title and starting with a given fragment.
prompt_template = (
"""Please generate a coherent continuation of the story "{story_name}".
Cut the story continuation into paragraphs, according to the logic of the narrative flow.
Include in the response only the story continuation once, without any additional text.
Start your story with the following fragment, then continue with your own text:
{fragment}
----\n
"""
)

Define functions for story segmentation.

In [ ]:
# Divide stories into segments of approximately equal length in number of tokens and return the first segment as a fragment and the number of tokens of the story.
def el_segment_story_frag(story, seg_length=100, save_inter_res=False, inter_res_seg_file=None):
    """
    Segments a story into smaller parts of approximately equal length in tokens.
    
    Args:
        story (str): The full text of the story.
        segment_length (int): The desired length of each segment in tokens.
        
    Returns:
        tuple: A tuple containing the token count of the story and the first segment as a fragment.
    """   
    # Split the story into tokens
    tokens = story.split()
    cnt_tok = len(tokens)  # Count the number of tokens in the story
    fragment = ""  # Initialize the fragment variable to be used in the prompt
    
    # Calculate the number of segments based on the desired segment length
    seg_count = math.ceil(len(tokens) / seg_length)
    
    # Length of each segment in tokens more equally distributed
    distrib_length = math.ceil(len(tokens) / seg_count) 
    
    # Create segments of approximately equal length
    # This will ensure that the segments are more evenly distributed in terms of token count
    segments = []
    for i in range(0, len(tokens), distrib_length):
        segment_tokens = tokens[i:i + distrib_length]
        segment = ' '.join(segment_tokens)
        if segment.strip():  # Add only non-empty segments
            segments.append(segment.strip())
    
    # Extract the first segment to use it as an opening fragment for the LLM-generated story
    if segments:
        fragment = segments[0]

    # If save_inter_res is True, save the intermediate results
    if save_inter_res:
        with open(inter_res_seg_file, "a", encoding="utf-8") as f:
            f.write(f"{ANN_SEP[0:4]} Opening fragment: {fragment}\n")

    # Return token count for the story and the first segment as a fragment
    return cnt_tok, fragment

In [ ]:
# Read an annotated story and return the first segment as a fragment and the number of tokens of the story.
def ann_segment_story_frag(story, save_inter_res = False, inter_res_seg_file=None):
    """
    Reads a text-tiling segmented story from a string and returns the segments.
    
    Args:
        story (str): The full text of the story.
        
    Returns:
        tuple: A tuple containing the token count of the story and the first segment as a fragment.
    """
    cnt_tok = 0  # Initialize the token count variable
    segments = [] # Initialize the list to hold segments
    fragment = ""  # Initialize the fragment variable to be used in the prompt
    
    # Split the story into parts based on the ANN_SEG label and ANN_SEP separator
    parts = story.split(ANN_SEP)
    for part in parts:
        # Skip the first line if it contains the ANN_SEG label
        if part.strip().startswith(ANN_SEG):
            part = '\n'.join(part.split('\n')[2:])  # Remove the line containing the annotation label
        if part.strip():  # Add only non-empty segments
            cnt_tok += len(part.strip().split())
            segments.append(part.strip())
    
    # Extract the first segment to use it as a fragment for the LLM-generated story
    if segments:
        fragment = segments[0]
    
        # If save_inter_res is True, save the intermediate results
    if save_inter_res:
        with open(inter_res_seg_file, "a", encoding="utf-8") as f:
            f.write(f"{ANN_SEP[0:4]} Opening fragment: {fragment}\n")

    return cnt_tok, fragment

Define the function for generating the stories and saving the results.

In [ ]:
# Process a set of stories, generate new stories starting with the first segment and save results as plain text files
def generate_stories(seg_method, save_inter_res):
    """
    Processes all stories in the input directory and saves the SGS results to the output directory.
    
    Args:
        seg_method (str): The method used for segmenting the stories ('equal-length', 'text-tiling', 'llm-episodes').
        save_inter_res (bool): Whether to save intermediate results to a file.
    """
    cnt_processed = 0 # Counter for processed files
    fn_seg_meth=FN_EQUAL_LENGTH # File name suffix for segmentation method, default is 'equal-length'
    inp = input # Default input path for whole stories
    out = output_el_start # Default output path for generated stories
    fragment = ""  # Initialize the fragment variable to 
    cnt_tok = 0  # Initialize the token count for the story

    # Select the input path and the file name suffix based on the segmentation method
    if seg_method == SEG_TEXT_TILING:
        inp = input_seg_tt
        out = output_tt_start
        fn_seg_meth = FN_TEXT_TILING
    # elif seg_method == SEG_LLM_EPISODE:
    #    inp = input_seg_llm
    #    out = output_llme_start
    #    fn_seg_meth = FN_LLM_EPISODE

    # Create a timestamp for the output files
    timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S') 

    # Extract the model name from the llm object (without the file extension)
    model_name = model_path.split('/')[-1].split('.gguf')[0]

    # File names for intermediate results
    if save_inter_res: # Create the intermediate results files if saving is enabled
        inter_res_seg_file = os.path.join(output_inter_res, f"{INTER_RES_SEG}_{fn_seg_meth}_{GEN_LABEL.lower()}_{timestamp}.txt")
        with open(inter_res_seg_file, "w", encoding="utf-8") as f:
            f.write(f"{INTER_RES_SEG.replace('_', ' ').capitalize()}: Generated stories by {model_name} from an opening segment: segmentation method: {seg_method}\n{ANN_SEP}\n")

    # Read the story files sequentially from the input folder.
    for file_path in Path(inp).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as input_file:
            story = input_file.read()
    
            # Print the name of the file being processed.
            curr_proc_str = f'Processing file: {file_path.name}'
            print(curr_proc_str)

            if save_inter_res: # Save the intermediate results to a file
                with open(inter_res_seg_file, "a", encoding="utf-8") as f:
                    f.write(f"{curr_proc_str}\n{ANN_SEP}\n")

            if (seg_method == SEG_TEXT_TILING):
                # Get the title of the story
                start = story.find(ANN_SEG + " 0:\n") + len(ANN_SEG + " 0:\n")
                end = story.find(ANN_SEP)
                story_name = story[start:end].strip()
                # Get the body of the story, excluding the title
                body = story[end+len(ANN_SEP):]
                # Segment the story into fragments using text-tiling and return the first segment as a fragment
                cnt_tok, fragment = ann_segment_story_frag(body, save_inter_res, inter_res_seg_file)
            else: # use equal-length segmentation
                # Get the title of the story
                story_name = story.split('\n')[0]
                # Get the body of the story, excluding the title
                body = story[story.find('\n')+1:]
                # Segment the story into fragments of approximately equal length and return the first segment as a fragment
                cnt_tok, fragment = el_segment_story_frag(body, seg_length=100, save_inter_res=save_inter_res, inter_res_seg_file=inter_res_seg_file)

            # Prepare the title for the generated story
            title = f"AI-generated {story_name}\n\n"
            
            # Prepare the prompt for the LLM
            prompt = f"""{prompt_template}{story_name}{fragment}"""

            # Generate a story using the LLM based on the prompt 
            response = llm(prompt, max_tokens=8192)["choices"][0]["text"].strip()
            response = response.replace('----\n\n', '')  # Replace the separator in the response with an empty string
            gen_story = title + fragment + " " + response

            if save_inter_res: # Save the intermediate results to a file
                with open(inter_res_seg_file, "a", encoding="utf-8") as f:
                    f.write(f"{ANN_SEP[0:4]} Token count original story: {cnt_tok}. Token count generated story: {len(gen_story.split())-len(title.split())}.\n{ANN_SEP}\n")

            # Clean the file_path name of timestamps, to shorten the file name
            clean_stem = re.sub(r'\d{8}_\d{6}', '#', file_path.stem)
            
            # Save the generated story to a text file
            gen_story_path = os.path.join(out, f"{clean_stem}_{fn_seg_meth}_{GEN_LABEL.lower()}_{model_name}_{timestamp}.txt")
            with open(gen_story_path, "w", encoding="utf-8") as f:
                f.write(gen_story)

            cnt_processed += 1
 
    # Print the number of files saved in the output folder
    end_proc_str = f'End processing: {cnt_processed} files saved to the {Path(out).name} folder.'
    print(end_proc_str)
    if save_inter_res:
        with open(inter_res_seg_file, "a", encoding="utf-8") as f:
            f.write(f"{end_proc_str}\n")

Validate input, read the starting fragments, generate new stories based on them and save the results.  

In [ ]:
# Choose the segmentation method
seg_method = SEG_EQUAL_LENGTH  # SEG_EQUAL_LENGTH or SEG_TEXT_TILING
# Set the flag to save intermediate results
save_inter_res = False  # Set to True to save intermediate results for segment check, False otherwise

# Validate the segmentation method and set the input path accordingly
if seg_method not in [SEG_EQUAL_LENGTH, SEG_TEXT_TILING]:
    raise ValueError("Invalid segmentation method. Choose from ", SEG_EQUAL_LENGTH, ", ", SEG_TEXT_TILING, ".")

# Process the stories with the specified SGS type, segmentation method and flag for saving intermediate results
generate_stories(seg_method=seg_method, save_inter_res=save_inter_res)